# Sub-Task 1: Fascia Detection — Architecture Comparison

Trains and compares **5 segmentation architectures** on ultrasound fascia detection:
- **U-Net (ResNet-50)** — CNN baseline with skip connections
- **U-Net++ (EfficientNet-B4)** — Nested dense skip connections
- **Attention U-Net (ResNet-50)** — Attention gates on skip connections
- **SegFormer-B5** — Pure transformer, no positional encoding
- **SwinUNet** — Swin Transformer encoder + U-Net decoder (medical-focused)

**Output mask classes:**
- `0` = Below fascia (deep region)
- `1` = Fascia band
- `2` = Above fascia (superficial region)

**Ground truth extraction:** Yellow pixels from Folder 3 (Simple Annotated) — yellow = fascia only in that folder.

In [ ]:
# ── GPU / Environment Check ───────────────────────────────────────────────────
import subprocess, torch

print('=' * 65)
print('ENVIRONMENT')
print('=' * 65)
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} | version {torch.version.cuda}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}    : {p.name}  |  {p.total_memory/1024**3:.1f} GB VRAM')

print()
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout)

In [ ]:
# ── Fix torchvision/torch mismatch, then install all deps ─────────────────────
#
# Root cause: torchvision was installed from PyPI (wrong build) instead of the
# official PyTorch wheel server.  The C++ torchvision::nms operator is absent,
# causing RuntimeError on import.
#
# Fix: force-reinstall torchvision from the same wheel server as PyTorch,
# matching the exact torch version and CUDA tag.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, torch

torch_ver = torch.__version__.split('+')[0]               # e.g. '2.6.0'
cuda_raw  = (torch.version.cuda or '').replace('.', '')   # e.g. '124'
cuda_tag  = f'cu{cuda_raw[:3]}' if cuda_raw else 'cpu'    # e.g. 'cu124'
index_url = f'https://download.pytorch.org/whl/{cuda_tag}'

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA tag : {cuda_tag}')
print(f'Wheel URL: {index_url}')
print('Re-installing torchvision from official wheel server …')

subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--force-reinstall', '--no-deps',
    'torchvision',
    '--index-url', index_url,
    '-q',
], check=True)
print('torchvision reinstalled.')

# ── All other packages ────────────────────────────────────────────────────────
pkgs = [
    'segmentation-models-pytorch',
    'transformers>=4.40.0',
    'timm>=1.0.0',
    'albumentations>=1.4.0',
    'opencv-python-headless',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'tqdm',
    'pandas',
    'scipy',
    'torchmetrics',
]
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=True)

print('\nAll packages ready.')
print('>>> RESTART THE KERNEL NOW, then re-run all cells from the top. <<<')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, cv2, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm
from scipy.ndimage import label as scipy_label
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
import timm
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerConfig,
)
from torchmetrics import JaccardIndex

warnings.filterwarnings('ignore')

# cuDNN 9.2.0 cannot initialise on Blackwell (SM120)
torch.backends.cudnn.enabled           = False
torch.backends.cudnn.benchmark         = False
torch.backends.cuda.matmul.allow_tf32  = True

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16
print(f'Using device : {DEVICE}')
print(f'AMP dtype    : {AMP_DTYPE}')


In [ ]:
# ── Global Config ─────────────────────────────────────────────────────────────
class CFG:
    # Paths  (notebook lives in Task_3/notebooks/  →  data is one level up)
    ROOT        = Path('../')
    RAW_DIR     = ROOT / 'Data' / '1 - Videos'
    ANN_DIR     = ROOT / 'Data' / '3 - Simple Annotated videos'
    OUT_DIR     = ROOT / 'output' / 'fascia'
    FRAMES_DIR  = OUT_DIR / 'frames'
    MASKS_DIR   = OUT_DIR / 'masks'
    CKPT_DIR    = OUT_DIR / 'checkpoints'
    PLOT_DIR    = OUT_DIR / 'plots'

    # Image
    IMG_SIZE    = 384          # all models use 384x384
    NUM_CLASSES = 3

    # Training
    EPOCHS      = 60
    BATCH_SIZE  = 24           # reduce to 16 if OOM
    LR          = 3e-4
    WEIGHT_DECAY= 1e-4
    NUM_WORKERS = 8
    SEED        = 42
    PATIENCE    = 15           # early stopping

    # Which videos go to validation (held-out, most different device/anatomy)
    VAL_VIDEOS  = ['Knee-Ankle-Seg1']

    # Class colours for visualisation
    CLS_COLORS  = {
        0: (0.15, 0.47, 0.71),   # blue  – below fascia
        1: (1.00, 0.85, 0.00),   # yellow – fascia band
        2: (0.17, 0.63, 0.17),   # green – above fascia
    }
    CLS_NAMES   = {0: 'Below fascia', 1: 'Fascia band', 2: 'Above fascia'}

# Create output dirs
for d in [CFG.FRAMES_DIR, CFG.MASKS_DIR, CFG.CKPT_DIR, CFG.PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)
print('Config ready. Output at:', CFG.OUT_DIR.resolve())

---
## 1  Data Extraction
Extract paired (frame, mask) from the processed and simple-annotated videos.

**Yellow detection** (HSV): H 18–42 · S > 70 · V > 70 → fascia lines only in Folder 3.

In [ ]:
# ── Mask Extraction Utilities ─────────────────────────────────────────────────

def detect_yellow_mask(bgr_frame):
    """Return binary mask of yellow pixels (fascia annotation lines)."""
    hsv = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2HSV)
    # Primary yellow range
    m1 = cv2.inRange(hsv, np.array([18, 70, 70]),  np.array([42, 255, 255]))
    # Exclude green (vein circles) that may bleed
    green = cv2.inRange(hsv, np.array([46, 60, 60]), np.array([90, 255, 255]))
    mask = cv2.bitwise_and(m1, cv2.bitwise_not(green))
    # Clean tiny noise
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    return mask


def mask_to_3class(yellow_mask):
    """
    Column-wise: find topmost and bottommost yellow pixel.
    0 = below fascia, 1 = fascia band, 2 = above fascia.
    Returns (mask_array, has_fascia_flag).
    """
    h, w = yellow_mask.shape
    out = np.zeros((h, w), dtype=np.uint8)
    has_fascia = False

    for x in range(w):
        col = yellow_mask[:, x]
        ys  = np.where(col > 0)[0]
        if len(ys) >= 2:
            y_top = int(ys.min())
            y_bot = int(ys.max())
            out[:y_top, x]          = 2   # above
            out[y_top:y_bot + 1, x] = 1   # band
            # below stays 0
            has_fascia = True
        elif len(ys) == 1:
            out[:int(ys[0]), x] = 2
            has_fascia = True

    return out, has_fascia


def validate_mask(mask):
    """Reject masks where fascia band is < 2 px tall on average (bad frame)."""
    band_cols = (mask == 1).sum(axis=0)  # pixels in class 1 per column
    return float(band_cols[band_cols > 0].mean()) > 2 if band_cols.any() else False


print('Mask utilities defined.')

In [ ]:
# ── Extract All Frames + Masks ────────────────────────────────────────────────
SAMPLE_STRIDE = 3   # save every Nth frame to avoid near-duplicate redundancy

metadata = []   # list of dicts {frame_path, mask_path, video, has_fascia}

raw_vids = sorted(CFG.RAW_DIR.glob('*.mp4')) + sorted(CFG.RAW_DIR.glob('*.MP4'))
print(f'Found {len(raw_vids)} videos in processed folder.')

for raw_path in raw_vids:
    stem     = raw_path.stem
    # Match annotated video (case-insensitive .mp4/.MP4)
    ann_path = CFG.ANN_DIR / raw_path.name
    if not ann_path.exists():
        ann_path = CFG.ANN_DIR / (raw_path.stem + '.mp4')
    if not ann_path.exists():
        print(f'  [SKIP] no annotated counterpart for {raw_path.name}'); continue

    cap_raw = cv2.VideoCapture(str(raw_path))
    cap_ann = cv2.VideoCapture(str(ann_path))
    total   = int(cap_raw.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'  Processing {stem}  ({total} frames) …')

    frame_dir = CFG.FRAMES_DIR / stem
    mask_dir  = CFG.MASKS_DIR  / stem
    frame_dir.mkdir(exist_ok=True)
    mask_dir.mkdir(exist_ok=True)

    fi = 0
    saved = 0
    while True:
        ret_r, frame = cap_raw.read()
        ret_a, ann   = cap_ann.read()
        if not ret_r or not ret_a:
            break

        if fi % SAMPLE_STRIDE != 0:
            fi += 1
            continue

        # Resize both to CFG.IMG_SIZE
        sz    = (CFG.IMG_SIZE, CFG.IMG_SIZE)
        frame = cv2.resize(frame, sz)
        ann   = cv2.resize(ann,   sz)

        # Extract mask
        ymask       = detect_yellow_mask(ann)
        seg_mask, has_f = mask_to_3class(ymask)
        valid       = validate_mask(seg_mask) if has_f else False

        fp = frame_dir / f'{fi:05d}.png'
        mp = mask_dir  / f'{fi:05d}.png'
        cv2.imwrite(str(fp), frame)
        cv2.imwrite(str(mp), seg_mask)

        metadata.append({
            'frame_path':  str(fp),
            'mask_path':   str(mp),
            'video':       stem,
            'frame_idx':   fi,
            'has_fascia':  has_f,
            'valid_mask':  valid,
        })
        saved += 1
        fi += 1

    cap_raw.release()
    cap_ann.release()
    print(f'    Saved {saved} frames.')

df = pd.DataFrame(metadata)
df.to_csv(CFG.OUT_DIR / 'metadata.csv', index=False)
print(f'\nTotal frames extracted : {len(df)}')
print(f'Frames with fascia     : {df.has_fascia.sum()}')
print(f'Valid masks            : {df.valid_mask.sum()}')
print(df.groupby('video')[['has_fascia','valid_mask']].sum())

In [ ]:
# ── Visualise: Sample Frames + Masks ─────────────────────────────────────────
def colorise_mask(mask):
    """Convert class mask (H,W) → RGB image for display."""
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    colors = {0: (40, 120, 180), 1: (255, 215, 0), 2: (44, 160, 44)}
    for cls, col in colors.items():
        rgb[mask == cls] = col
    return rgb


def overlay_mask(frame_bgr, mask, alpha=0.45):
    """Blend colourised mask over the original frame."""
    frame_rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mask_rgb   = colorise_mask(mask)
    return (frame_rgb * (1 - alpha) + mask_rgb * alpha).astype(np.uint8)


# Pick one representative frame per video
sample_rows = df[df.valid_mask].groupby('video').apply(lambda g: g.iloc[len(g)//2])
n = len(sample_rows)
fig, axes = plt.subplots(n, 3, figsize=(14, 4.5 * n))
if n == 1: axes = axes[np.newaxis]

for i, (_, row) in enumerate(sample_rows.iterrows()):
    frame = cv2.imread(row.frame_path)
    mask  = cv2.imread(row.mask_path,  cv2.IMREAD_GRAYSCALE)

    axes[i, 0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); axes[i, 0].set_title(f'{row.video}\nRaw frame')
    axes[i, 1].imshow(colorise_mask(mask));                    axes[i, 1].set_title('3-class mask')
    axes[i, 2].imshow(overlay_mask(frame, mask));              axes[i, 2].set_title('Overlay')
    for ax in axes[i]: ax.axis('off')

patches = [mpatches.Patch(color=np.array(c)/255, label=n)
           for n, c in [('Below fascia',(40,120,180)),('Fascia band',(255,215,0)),('Above fascia',(44,160,44))]]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=12)
plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'sample_frames_masks.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualise: Class Distribution ────────────────────────────────────────────
# Sample ~500 valid masks and count pixel-level class distribution
sample_df  = df[df.valid_mask].sample(min(500, df.valid_mask.sum()), random_state=CFG.SEED)
class_counts = np.zeros(3, dtype=np.int64)
for _, row in sample_df.iterrows():
    m = cv2.imread(row.mask_path, cv2.IMREAD_GRAYSCALE)
    for c in range(3):
        class_counts[c] += (m == c).sum()

labels = ['Below fascia (0)', 'Fascia band (1)', 'Above fascia (2)']
colors = ['#2878b0', '#ffd700', '#2ca02c']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(labels, class_counts, color=colors)
axes[0].set_title('Pixel count per class'); axes[0].set_ylabel('Pixels')
axes[0].tick_params(axis='x', rotation=15)

axes[1].pie(class_counts, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class distribution')
plt.suptitle('Dataset class balance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
for l, c in zip(labels, class_counts):
    print(f'  {l}: {c:>10,} px  ({100*c/class_counts.sum():.1f}%)')

---
## 2  Dataset & DataLoader

In [ ]:
# ── Augmentation Pipelines ────────────────────────────────────────────────────
TRAIN_TRANSFORMS = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=15, p=0.5),
    A.RandomResizedCrop(size=(CFG.IMG_SIZE, CFG.IMG_SIZE),
                        scale=(0.75, 1.0), ratio=(0.9, 1.1), p=0.4),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 50), p=1),
        A.MultiplicativeNoise(multiplier=(0.85, 1.15), p=1),
        A.ISONoise(p=1),
    ], p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1),
        A.MedianBlur(blur_limit=3, p=1),
        A.MotionBlur(blur_limit=5, p=1),
    ], p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.25, p=0.6),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3),
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 20),
                    hole_width_range=(1, 20), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

VAL_TRANSFORMS = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

print('Augmentation pipelines defined.')


In [ ]:
# ── Dataset Class ─────────────────────────────────────────────────────────────
class FasciaDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe[dataframe.valid_mask].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = cv2.imread(row.frame_path)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(row.mask_path, cv2.IMREAD_GRAYSCALE).astype(np.int64)

        if self.transform:
            aug  = self.transform(image=img, mask=mask.astype(np.uint8))
            img  = aug['image']
            mask = aug['mask'].long()

        return img, mask


# ── Random per-video split (20% val) ─────────────────────────────────────────
def split_by_random(df, val_frac=0.2, seed=42):
    train_rows, val_rows = [], []
    for vid in df['video'].unique():
        grp   = df[df['video'] == vid].sample(frac=1, random_state=seed).reset_index(drop=True)
        n_val = max(1, int(len(grp) * val_frac))
        val_rows.append(grp.iloc[:n_val])
        train_rows.append(grp.iloc[n_val:])
    return pd.concat(train_rows).reset_index(drop=True), pd.concat(val_rows).reset_index(drop=True)

train_df, val_df = split_by_random(df[df.valid_mask].copy(), val_frac=0.2)

train_ds = FasciaDataset(train_df, TRAIN_TRANSFORMS)
val_ds   = FasciaDataset(val_df,   VAL_TRANSFORMS)

print(f'Train samples : {len(train_ds):>5}')
print(f'Val   samples : {len(val_ds):>5}')
print('Videos in train:', train_df.video.unique().tolist())
print('Videos in val  :', val_df.video.unique().tolist())


In [ ]:
# ── Visualise Augmented Samples ───────────────────────────────────────────────
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denorm(t):
    """Denormalise tensor (3,H,W) → numpy uint8 RGB."""
    arr = t.permute(1, 2, 0).numpy()
    arr = (arr * STD + MEAN).clip(0, 1)
    return (arr * 255).astype(np.uint8)

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(2.5 * n_show, 6))
batch_imgs, batch_masks = next(iter(train_loader))

for j in range(n_show):
    axes[0, j].imshow(denorm(batch_imgs[j]));          axes[0, j].axis('off')
    axes[1, j].imshow(colorise_mask(batch_masks[j].numpy())); axes[1, j].axis('off')
axes[0, 0].set_ylabel('Image', fontsize=12)
axes[1, 0].set_ylabel('Mask',  fontsize=12)
plt.suptitle('Augmented Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'augmented_samples.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3  Model Architectures
All models use ImageNet pre-trained weights. Final output: `(B, num_classes, H, W)`.

In [ ]:
# ── Shared Helper Blocks ──────────────────────────────────────────────────────
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print('Helper blocks defined.')

In [ ]:
# ── Model 1: U-Net (ResNet-50) ────────────────────────────────────────────────
def build_unet_resnet50():
    model = smp.Unet(
        encoder_name    = 'resnet50',
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = CFG.NUM_CLASSES,
        activation      = None,   # raw logits, loss handles softmax
    )
    return model

m1 = build_unet_resnet50()
t, tr = count_params(m1)
print(f'[1] U-Net ResNet-50     | Total: {t/1e6:.1f}M | Trainable: {tr/1e6:.1f}M')

In [ ]:
# ── Model 2: U-Net++ (EfficientNet-B4) ───────────────────────────────────────
def build_unetpp_efficientnet():
    model = smp.UnetPlusPlus(
        encoder_name    = 'efficientnet-b4',
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = CFG.NUM_CLASSES,
        activation      = None,
    )
    return model

m2 = build_unetpp_efficientnet()
t, tr = count_params(m2)
print(f'[2] U-Net++ EfficientB4 | Total: {t/1e6:.1f}M | Trainable: {tr/1e6:.1f}M')

In [ ]:
# ── Model 3: Attention U-Net (ResNet-50) ──────────────────────────────────────
def build_att_unet():
    model = smp.Unet(
        encoder_name     = 'resnet50',
        encoder_weights  = 'imagenet',
        decoder_attention_type = 'scse',   # Squeeze-Excitation channel + spatial
        in_channels      = 3,
        classes          = CFG.NUM_CLASSES,
        activation       = None,
    )
    return model

m3 = build_att_unet()
t, tr = count_params(m3)
print(f'[3] Att U-Net ResNet-50 | Total: {t/1e6:.1f}M | Trainable: {tr/1e6:.1f}M')

In [ ]:
# ── Model 4: SegFormer-B5 ─────────────────────────────────────────────────────
class SegFormerWrapper(nn.Module):
    """
    Wraps HuggingFace SegformerForSemanticSegmentation so it outputs
    (B, num_classes, H, W) matching the rest of the training loop.
    HF model outputs logits at H/4 × W/4; we bilinear-upsample to input size.
    """
    def __init__(self, num_classes=3, img_size=384):
        super().__init__()
        self.img_size = img_size
        cfg = SegformerConfig.from_pretrained('nvidia/mit-b5')
        cfg.num_labels   = num_classes
        cfg.id2label     = {i: str(i) for i in range(num_classes)}
        cfg.label2id     = {str(i): i for i in range(num_classes)}
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            'nvidia/mit-b5',
            config           = cfg,
            ignore_mismatched_sizes = True,
        )

    def forward(self, x):
        out    = self.model(pixel_values=x)
        logits = out.logits                            # (B, C, H/4, W/4)
        return F.interpolate(logits,
                             size=(self.img_size, self.img_size),
                             mode='bilinear', align_corners=False)


m4 = SegFormerWrapper(num_classes=CFG.NUM_CLASSES, img_size=CFG.IMG_SIZE)
t, tr = count_params(m4)
print(f'[4] SegFormer-B5        | Total: {t/1e6:.1f}M | Trainable: {tr/1e6:.1f}M')

In [ ]:
# ── Model 5: SwinUNet (Swin-Base encoder + U-Net decoder) ─────────────────────
# Uses timm's Swin-Base pretrained at 384×384 as encoder,
# with a custom multi-scale U-Net decoder for full-resolution output.

class SwinUNetDecoder(nn.Module):
    def __init__(self, enc_channels, num_classes):
        super().__init__()
        c = enc_channels   # [128, 256, 512, 1024] for swin_base
        self.up3 = ConvBnRelu(c[3] + c[2], c[2])
        self.up2 = ConvBnRelu(c[2] + c[1], c[1])
        self.up1 = ConvBnRelu(c[1] + c[0], c[0])
        self.up0 = ConvBnRelu(c[0],         64)
        self.head = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, feats):
        # feats: list of (B, H_i, W_i, C_i) from timm Swin — convert to NCHW
        f = [x.permute(0, 3, 1, 2).contiguous() for x in feats]
        # f[0]: stride-4, f[1]: stride-8, f[2]: stride-16, f[3]: stride-32

        x = F.interpolate(f[3], size=f[2].shape[2:], mode='bilinear', align_corners=False)
        x = self.up3(torch.cat([x, f[2]], dim=1))

        x = F.interpolate(x, size=f[1].shape[2:], mode='bilinear', align_corners=False)
        x = self.up2(torch.cat([x, f[1]], dim=1))

        x = F.interpolate(x, size=f[0].shape[2:], mode='bilinear', align_corners=False)
        x = self.up1(torch.cat([x, f[0]], dim=1))

        # Upsample from stride-4 back to original resolution
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False)
        x = self.up0(x)
        return self.head(x)


class SwinUNet(nn.Module):
    def __init__(self, num_classes=3, img_size=384):
        super().__init__()
        # swin_base_patch4_window12_384 is pretrained on ImageNet-22K at 384×384
        self.encoder = timm.create_model(
            'swin_base_patch4_window12_384',
            pretrained=True,
            features_only=True,
            out_indices=(0, 1, 2, 3),
        )
        enc_channels = [128, 256, 512, 1024]   # swin_base channel widths
        self.decoder  = SwinUNetDecoder(enc_channels, num_classes)

    def forward(self, x):
        feats = self.encoder(x)   # list of NHWC tensors
        return self.decoder(feats)


m5 = SwinUNet(num_classes=CFG.NUM_CLASSES, img_size=CFG.IMG_SIZE)
t, tr = count_params(m5)
print(f'[5] SwinUNet (Swin-Base)| Total: {t/1e6:.1f}M | Trainable: {tr/1e6:.1f}M')

In [ ]:
# ── Forward-pass smoke test + parameter comparison chart ──────────────────────
MODEL_REGISTRY = {
    'UNet-ResNet50':     build_unet_resnet50,
    'UNet++-EffNetB4':   build_unetpp_efficientnet,
    'AttUNet-ResNet50':  build_att_unet,
    'SegFormer-B5':      lambda: SegFormerWrapper(CFG.NUM_CLASSES, CFG.IMG_SIZE),
    'SwinUNet':          lambda: SwinUNet(CFG.NUM_CLASSES, CFG.IMG_SIZE),
}

dummy   = torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)
param_M = {}

for name, builder in MODEL_REGISTRY.items():
    m = builder().eval()
    with torch.no_grad():
        out = m(dummy)
    assert out.shape == (2, CFG.NUM_CLASSES, CFG.IMG_SIZE, CFG.IMG_SIZE), f'{name}: bad shape {out.shape}'
    total, _ = count_params(m)
    param_M[name] = total / 1e6
    print(f'{name:<25}: output {tuple(out.shape)}  |  {total/1e6:.1f}M params')
    del m

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(list(param_M.keys()), list(param_M.values()),
               color=['#4878cf','#6acc65','#d65f5f','#b47cc7','#c4ad66'])
ax.bar_label(bars, fmt='%.1f M', padding=4)
ax.set_xlabel('Parameters (M)')
ax.set_title('Model Size Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'model_params.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4  Loss, Metrics & Trainer

In [ ]:
# ── Combined Loss: Dice + Cross-Entropy ───────────────────────────────────────
class DiceCELoss(nn.Module):
    def __init__(self, num_classes=3, dice_w=0.6, ce_w=0.4,
                 class_weights=None, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.dice_w  = dice_w
        self.ce_w    = ce_w
        self.smooth  = smooth
        w = torch.tensor(class_weights, dtype=torch.float32) if class_weights else None
        self.ce = nn.CrossEntropyLoss(weight=w)

    def dice_loss(self, probs, targets):
        B, C, H, W = probs.shape
        t_one_hot  = F.one_hot(targets, C).permute(0, 3, 1, 2).float()
        inter = (probs * t_one_hot).sum(dim=(0, 2, 3))
        union = probs.sum(dim=(0, 2, 3)) + t_one_hot.sum(dim=(0, 2, 3))
        dice  = (2. * inter + self.smooth) / (union + self.smooth)
        return 1. - dice.mean()

    def forward(self, logits, targets):
        probs   = torch.softmax(logits, dim=1)
        ce_loss = self.ce(logits, targets)
        dl      = self.dice_loss(probs, targets)
        return self.dice_w * dl + self.ce_w * ce_loss


CRITERION = DiceCELoss(
    num_classes   = CFG.NUM_CLASSES,
    class_weights = [1.0, 10.0, 1.0],   # 10x weight on thin fascia band
).to(DEVICE)
print('Loss function ready.')


In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(preds_list, targets_list, num_classes=3):
    _zero = {k: 0.0 for k in ['iou_below','iou_fascia','iou_above','mIoU','dice_fascia','mean_dice']}
    if not preds_list:
        return _zero
    try:
        all_preds   = torch.cat(preds_list,   dim=0).cpu()
        all_targets = torch.cat(targets_list, dim=0).cpu()

        jaccard = JaccardIndex(task='multiclass', num_classes=num_classes, average='none')
        per_iou = jaccard(all_preds, all_targets).detach().cpu().numpy()

        per_dice = np.zeros(num_classes, dtype=np.float32)
        for c in range(num_classes):
            p = (all_preds == c).float()
            t = (all_targets == c).float()
            per_dice[c] = (2*(p*t).sum() / (p.sum() + t.sum() + 1e-6)).item()

        return {
            'iou_below':   float(per_iou[0]),
            'iou_fascia':  float(per_iou[1]),
            'iou_above':   float(per_iou[2]),
            'mIoU':        float(per_iou.mean()),
            'dice_fascia': float(per_dice[1]),
            'mean_dice':   float(per_dice.mean()),
        }
    except Exception:
        import traceback; traceback.print_exc()
        return _zero


def boundary_f1(pred_mask, gt_mask, cls=1, dilation=2):
    import cv2 as _cv2
    k = np.ones((3, 3), np.uint8)
    def boundary(m):
        m8 = (m == cls).astype(np.uint8) * 255
        return _cv2.dilate(m8, k, iterations=dilation) - _cv2.erode(m8, k, iterations=dilation)
    pb = boundary(pred_mask.numpy()) > 0
    gb = boundary(gt_mask.numpy())   > 0
    tp = (pb & gb).sum()
    prec = tp / (pb.sum() + 1e-6)
    rec  = tp / (gb.sum() + 1e-6)
    return 2 * prec * rec / (prec + rec + 1e-6)


print('Metrics ready.')


In [ ]:
# ── Trainer Class ─────────────────────────────────────────────────────────────
class Trainer:
    def __init__(self, model, name, epochs=CFG.EPOCHS, lr=CFG.LR):
        self.model   = model.to(DEVICE)
        self.name    = name
        self.epochs  = epochs
        # GradScaler(device=) is the new API (device kwarg added in PyTorch 2.4)
        self.scaler  = GradScaler(device='cuda')

        # SegFormer: lower LR for pretrained encoder, full LR for decode head
        if isinstance(model, SegFormerWrapper):
            params = [
                {'params': model.model.segformer.parameters(),   'lr': lr * 0.1},
                {'params': model.model.decode_head.parameters(), 'lr': lr},
            ]
        else:
            params = model.parameters()

        self.optim = torch.optim.AdamW(params, lr=lr, weight_decay=CFG.WEIGHT_DECAY)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                         self.optim, T_max=epochs, eta_min=1e-6)

        self.history          = defaultdict(list)
        self.best_miou        = 0.0
        self.patience_counter = 0
        self.ckpt_path        = CFG.CKPT_DIR / f'{name}_best.pth'

    # ── one epoch ────────────────────────────────────────────────────────────
    def _run_epoch(self, loader, train=True):
        self.model.train() if train else self.model.eval()
        running_loss = 0.0
        all_preds, all_targets = [], []

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for imgs, masks in loader:
                imgs  = imgs.to(DEVICE, non_blocking=True)
                masks = masks.to(DEVICE, non_blocking=True)

                # autocast(device_type=) is the correct new-API call
                with autocast(device_type='cuda', dtype=AMP_DTYPE):
                    logits = self.model(imgs)
                    loss   = CRITERION(logits, masks)

                if train:
                    self.optim.zero_grad(set_to_none=True)
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.optim)
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.scaler.step(self.optim)
                    self.scaler.update()

                running_loss += loss.item()
                preds = logits.argmax(dim=1).cpu()
                all_preds.append(preds)
                all_targets.append(masks.cpu())

        avg_loss = running_loss / len(loader)
        metrics  = compute_metrics(all_preds, all_targets)
        return avg_loss, metrics

    # ── full training loop ────────────────────────────────────────────────────
    def fit(self, train_loader, val_loader):
        print(f'\n{"═"*60}')
        print(f'  Training: {self.name}')
        print(f'{"═"*60}')

        for epoch in range(1, self.epochs + 1):
            t0 = time.time()
            tr_loss, tr_m = self._run_epoch(train_loader, train=True)
            va_loss, va_m = self._run_epoch(val_loader,   train=False)
            self.sched.step()

            # Log history
            self.history['tr_loss'].append(tr_loss)
            self.history['va_loss'].append(va_loss)
            for k, v in va_m.items():
                self.history[f'va_{k}'].append(v)
            for k, v in tr_m.items():
                self.history[f'tr_{k}'].append(v)

            miou = va_m['mIoU']
            dt   = time.time() - t0
            print(f'  Ep {epoch:3d}/{self.epochs}  '
                  f'loss {tr_loss:.4f}/{va_loss:.4f}  '
                  f'mIoU {tr_m["mIoU"]:.4f}/{miou:.4f}  '
                  f'FasciaDice {va_m["dice_fascia"]:.4f}  '
                  f'[{dt:.1f}s]')

            if miou > self.best_miou:
                self.best_miou = miou
                torch.save(self.model.state_dict(), self.ckpt_path)
                self.patience_counter = 0
                print(f'    ✓ New best mIoU: {miou:.4f} — saved.')
            else:
                self.patience_counter += 1
                if self.patience_counter >= CFG.PATIENCE:
                    print(f'  Early stopping at epoch {epoch}.')
                    break

        print(f'  Best val mIoU: {self.best_miou:.4f}')
        return self.history

print('Trainer class ready.')

---
## 5  Train All Models

In [ ]:
import gc

gc.collect()
torch.cuda.synchronize()
torch.cuda.empty_cache()

# cuDNN 9.2.0 cannot initialise on Blackwell (SM120) — must stay disabled
torch.backends.cudnn.enabled           = False
torch.backends.cudnn.benchmark         = False
torch.backends.cuda.matmul.allow_tf32  = True

AMP_DTYPE = torch.bfloat16   # bfloat16 is natively fast on Blackwell without cuDNN

bs = 16
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False,
                          num_workers=4, pin_memory=True)

model   = build_unet_resnet50()
trainer = Trainer(model, name='UNet-ResNet50', epochs=120, unfreeze_epoch=15)
hist    = trainer.fit(train_loader, val_loader)

all_histories = {'UNet-ResNet50': dict(hist)}
with open(Path(CFG.OUT_DIR) / 'training_histories.json', 'w') as f:
    json.dump(all_histories, f, indent=2)

gc.collect()
torch.cuda.empty_cache()
print('Done.')


---
## 6  Results & Visualisation

In [ ]:
# ── Plot Training Curves ──────────────────────────────────────────────────────
COLORS = ['#4878cf', '#6acc65', '#d65f5f', '#b47cc7', '#c4ad66']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for (name, hist), col in zip(all_histories.items(), COLORS):
    e = range(1, len(hist['va_loss']) + 1)
    axes[0].plot(e, hist['tr_loss'], '--', color=col, alpha=0.5)
    axes[0].plot(e, hist['va_loss'], '-',  color=col, label=name)
    axes[1].plot(e, hist['va_mIoU'],        '-', color=col, label=name)
    axes[2].plot(e, hist['va_dice_fascia'], '-', color=col, label=name)

axes[0].set_title('Loss (dashed=train, solid=val)', fontweight='bold')
axes[1].set_title('Val mIoU',   fontweight='bold')
axes[2].set_title('Val Fascia-Band Dice', fontweight='bold')

for ax in axes:
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'training_curves.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final Metrics Table ───────────────────────────────────────────────────────
# Reload best checkpoints and evaluate on full val set

def evaluate_checkpoint(model_name, builder):
    ckpt  = CFG.CKPT_DIR / f'{model_name}_best.pth'
    model = builder().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()

    all_preds, all_targets, bf1_scores = [], [], []

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            with autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = model(imgs)
            preds = logits.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_targets.append(masks)
            for p, t in zip(preds, masks):
                bf1_scores.append(boundary_f1(p, t, cls=1))

    del model
    torch.cuda.empty_cache()
    m = compute_metrics(all_preds, all_targets)
    m['boundary_f1'] = float(np.mean(bf1_scores))
    return m


rows = []
for name, builder in MODEL_REGISTRY.items():
    print(f'Evaluating {name} …')
    m = evaluate_checkpoint(name, builder)
    rows.append({'Model': name, **m})

results_df = pd.DataFrame(rows).set_index('Model')
results_df = results_df[['mIoU', 'iou_fascia', 'dice_fascia',
                          'boundary_f1', 'iou_below', 'iou_above']]
results_df = results_df.round(4).sort_values('mIoU', ascending=False)
results_df.to_csv(CFG.OUT_DIR / 'final_metrics.csv')

print('\n' + '='*70)
print('FINAL METRICS (sorted by mIoU)')
print('='*70)
print(results_df.to_string())

In [ ]:
# ── Metrics Bar Charts ────────────────────────────────────────────────────────
metrics_to_plot = ['mIoU', 'iou_fascia', 'dice_fascia', 'boundary_f1']
titles = ['Mean IoU', 'Fascia-Band IoU', 'Fascia-Band Dice', 'Boundary F1']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
x = np.arange(len(results_df))

for ax, met, title in zip(axes, metrics_to_plot, titles):
    vals  = results_df[met].values
    bars  = ax.bar(x, vals, color=COLORS[:len(x)], edgecolor='k', linewidth=0.5)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(results_df.index, rotation=25, ha='right', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Model Comparison — Validation Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'metrics_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visual Predictions: All 5 Models Side-by-Side ─────────────────────────────
val_valid       = val_df[val_df.valid_mask].reset_index(drop=True)
indices         = [0, len(val_valid)//4, len(val_valid)//2, 3*len(val_valid)//4]
sample_rows_vis = [val_valid.iloc[i] for i in indices]
model_names     = list(MODEL_REGISTRY.keys())
n_rows, n_cols  = len(sample_rows_vis), 2 + len(model_names)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.5 * n_rows))

# Load all checkpoints once
loaded_models = {}
for mname, builder in MODEL_REGISTRY.items():
    m = builder().to(DEVICE)
    m.load_state_dict(torch.load(CFG.CKPT_DIR / f'{mname}_best.pth', map_location=DEVICE))
    m.eval()
    loaded_models[mname] = m

for ri, row in enumerate(sample_rows_vis):
    frame     = cv2.imread(row.frame_path)
    gt        = cv2.imread(row.mask_path, cv2.IMREAD_GRAYSCALE)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    axes[ri, 0].imshow(frame_rgb)
    axes[ri, 0].set_title('Input'        if ri == 0 else '')
    axes[ri, 1].imshow(colorise_mask(gt))
    axes[ri, 1].set_title('Ground Truth' if ri == 0 else '')

    aug = VAL_TRANSFORMS(image=frame_rgb, mask=gt.astype(np.uint8))
    inp = aug['image'].unsqueeze(0).to(DEVICE)

    for ci, mname in enumerate(model_names, start=2):
        with torch.no_grad(), autocast(device_type='cuda', dtype=AMP_DTYPE):
            pred = loaded_models[mname](inp).argmax(dim=1).squeeze().cpu().numpy()
        axes[ri, ci].imshow(colorise_mask(pred.astype(np.uint8)))
        axes[ri, ci].set_title(mname if ri == 0 else '', fontsize=8)

for ax in axes.flatten():
    ax.axis('off')

plt.suptitle('Prediction Comparison — Validation Frames', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'visual_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

for m in loaded_models.values():
    del m
torch.cuda.empty_cache()

In [ ]:
# ── Best Model: Overlay + Polynomial Curve Fit ────────────────────────────────
from numpy.polynomial import polynomial as P


def extract_fascia_curves(seg_mask, poly_degree=3):
    """
    Extract upper and lower fascia boundary curves from a 3-class mask.
    Returns (x_pts, y_upper, y_lower) as numpy float arrays,
    or (None, None, None) if the band is too sparse.
    """
    band = (seg_mask == 1).astype(np.uint8)
    h, w = band.shape
    xs, y_tops, y_bots = [], [], []

    for x in range(w):
        ys = np.where(band[:, x] > 0)[0]
        if len(ys) >= 2:
            xs.append(x)
            y_tops.append(int(ys.min()))
            y_bots.append(int(ys.max()))

    if len(xs) < poly_degree + 1:
        return None, None, None

    xs      = np.array(xs, float)
    xs_norm = xs / w
    c_top   = np.polyfit(xs_norm, y_tops, poly_degree)
    c_bot   = np.polyfit(xs_norm, y_bots, poly_degree)

    x_all  = np.arange(w, dtype=float)
    x_norm = x_all / w
    return x_all, np.polyval(c_top, x_norm), np.polyval(c_bot, x_norm)


# ── Load best model ───────────────────────────────────────────────────────────
best_name  = results_df.index[0]
print(f'Best model : {best_name}  (mIoU = {results_df.loc[best_name, "mIoU"]:.4f})')

best_model = MODEL_REGISTRY[best_name]().to(DEVICE)
best_model.load_state_dict(
    torch.load(CFG.CKPT_DIR / f'{best_name}_best.pth', map_location=DEVICE))
best_model.eval()

# ── Show 6 val frames: mask overlay + curve fit + GT curves ──────────────────
n_show = min(6, len(val_valid))
idxs   = np.linspace(0, len(val_valid) - 1, n_show, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(16, 11))
axes = axes.flatten()

for i, idx in enumerate(idxs):
    row       = val_valid.iloc[idx]
    frame     = cv2.imread(row.frame_path)
    gt        = cv2.imread(row.mask_path, cv2.IMREAD_GRAYSCALE)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    aug = VAL_TRANSFORMS(image=frame_rgb, mask=gt.astype(np.uint8))
    inp = aug['image'].unsqueeze(0).to(DEVICE)

    with torch.no_grad(), autocast(device_type='cuda', dtype=AMP_DTYPE):
        pred = best_model(inp).argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8)

    x_pts, y_up, y_lo = extract_fascia_curves(pred)
    xg,    yu_g, yl_g = extract_fascia_curves(gt)

    axes[i].imshow(frame_rgb)
    axes[i].imshow(colorise_mask(pred), alpha=0.30)
    if x_pts is not None:
        axes[i].plot(x_pts, y_up, 'y-',  lw=2.5, label='Pred upper')
        axes[i].plot(x_pts, y_lo, 'y--', lw=2.5, label='Pred lower')
    if xg is not None:
        axes[i].plot(xg, yu_g, 'g-',  lw=1.5, alpha=0.8, label='GT upper')
        axes[i].plot(xg, yl_g, 'g--', lw=1.5, alpha=0.8, label='GT lower')

    axes[i].set_title(f'Frame {row.frame_idx}  |  {row.video}', fontsize=9)
    axes[i].axis('off')
    if i == 0:
        axes[i].legend(fontsize=7, loc='lower right')

plt.suptitle(f'Best Model ({best_name}): Segmentation + Polynomial Curve Fit',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'best_model_curves.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Best Model: Temporal Fascia Curve Stability ───────────────────────────────
# Run on every saved frame of the validation video and plot y-position over time

val_vid_name = CFG.VAL_VIDEOS[0]
vid_frames   = val_valid[val_valid.video == val_vid_name].sort_values('frame_idx')
print(f'Running {best_name} on {len(vid_frames)} frames of {val_vid_name} …')

y_upper_seq, y_lower_seq, frame_ids = [], [], []

for _, row in tqdm(vid_frames.iterrows(), total=len(vid_frames)):
    frame     = cv2.imread(row.frame_path)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    aug = VAL_TRANSFORMS(image=frame_rgb,
                         mask=np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), np.uint8))
    inp = aug['image'].unsqueeze(0).to(DEVICE)

    with torch.no_grad(), autocast(device_type='cuda', dtype=AMP_DTYPE):
        pred = best_model(inp).argmax(1).squeeze().cpu().numpy().astype(np.uint8)

    x_pts, y_up, y_lo = extract_fascia_curves(pred)
    mid = CFG.IMG_SIZE // 2
    y_upper_seq.append(float(y_up[mid]) if y_up is not None else float('nan'))
    y_lower_seq.append(float(y_lo[mid]) if y_lo is not None else float('nan'))
    frame_ids.append(row.frame_idx)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(frame_ids, y_upper_seq, '-',  color='gold',       lw=1.8, label='Upper fascia y')
ax.plot(frame_ids, y_lower_seq, '--', color='darkorange',  lw=1.8, label='Lower fascia y')
ax.fill_between(frame_ids, y_upper_seq, y_lower_seq,
                alpha=0.15, color='yellow', label='Fascia band')
ax.invert_yaxis()
ax.set_xlabel('Frame index')
ax.set_ylabel('y pixel (0 = top of image)')
ax.set_title(f'Temporal Fascia Position — {val_vid_name}  |  {best_name}',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG.PLOT_DIR / 'temporal_fascia_curve.png', dpi=130, bbox_inches='tight')
plt.show()

del best_model
torch.cuda.empty_cache()

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('=' * 65)
print('FASCIA DETECTION — FINAL SUMMARY')
print('=' * 65)
print(results_df[['mIoU','iou_fascia','dice_fascia','boundary_f1']]
      .sort_values('mIoU', ascending=False).to_string())
print()
best = results_df.index[0]
print(f'Winner : {best}')
print(f'  mIoU           : {results_df.loc[best,"mIoU"]:.4f}')
print(f'  Fascia-band IoU: {results_df.loc[best,"iou_fascia"]:.4f}')
print(f'  Fascia Dice    : {results_df.loc[best,"dice_fascia"]:.4f}')
print(f'  Boundary F1    : {results_df.loc[best,"boundary_f1"]:.4f}')
print()
print(f'Best checkpoint: {CFG.CKPT_DIR / (best + "_best.pth")}')
print('All plots saved to:', CFG.PLOT_DIR.resolve())